# 📊 Data Analysis Notebook

---

## Objective
Explore the cleaned food price data to discover patterns, trends, and insights.

## Questions We'll Answer
1. 📈 How have food prices changed over time?
2. 🌍 Which countries have the highest/lowest inflation?
3. 📅 Are there seasonal patterns in food prices?
4. 🔗 What factors are related to each other?

## Prerequisites
Run `Data_Cleaning.ipynb` first to generate the cleaned dataset.

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully!")

---
## Step 2: Load Data

In [ ]:
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print("✅ Data Loaded!")
print(f"📊 Size: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📅 Period: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
print(f"🌍 Countries: {df['country'].nunique()}")

In [ ]:
df.head()

---
## Step 3: Summary Statistics

In [ ]:
numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
df[numerical_cols].describe().round(3)

In [ ]:
print("📊 KEY INSIGHTS")
print("="*50)
print(f"Average price index: {df['close'].mean():.2f}")
print(f"Average inflation: {df['inflation'].mean():.2f}%")
print(f"Highest inflation: {df['inflation'].max():.2f}%")
print(f"Lowest inflation: {df['inflation'].min():.2f}%")

---
## Step 4: Country Rankings

In [ ]:
print("🔥 TOP 5 COUNTRIES - HIGHEST INFLATION")
print("="*50)
top5 = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)
for rank, (country, val) in enumerate(top5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

print("\n🌟 TOP 5 COUNTRIES - LOWEST INFLATION")
print("="*50)
bottom5 = df.groupby('country')['inflation'].mean().sort_values().head(5)
for rank, (country, val) in enumerate(bottom5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

---
## Step 5: Distribution Analysis

In [ ]:
os.makedirs('../outputs/figures', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Distribution of Food Prices', fontweight='bold')
axes[0, 0].set_xlabel('Price Index')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Mean: {df["close"].mean():.2f}')
axes[0, 0].legend()

# Inflation distribution
axes[0, 1].hist(df['inflation'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Inflation', fontweight='bold')
axes[0, 1].set_xlabel('Inflation Rate (%)')
axes[0, 1].axvline(0, color='black', linestyle='-', label='Zero')
axes[0, 1].legend()

# Volatility distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('Distribution of Price Volatility', fontweight='bold')
axes[1, 0].set_xlabel('Price Range')

# Inflation by year
sns.boxplot(data=df[df['inflation'].notna()], x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation by Year', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/distribution_analysis.png")

---
## Step 6: Time Series Analysis

In [ ]:
monthly_avg = df.groupby('date').agg({'close': 'mean', 'inflation': 'mean'}).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('Global Average Food Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Price Index')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('Global Average Inflation Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Inflation (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/time_series_analysis.png")

---
## Step 7: Country Comparison

In [ ]:
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]
country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('Average Inflation by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Inflation (%)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/country_comparison.png")

---
## Step 8: Correlation Analysis

In [ ]:
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range']
correlation_matrix = df[correlation_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/correlation_matrix.png")

---
## Step 9: Seasonal Patterns

In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg_season = df.groupby('month')['inflation'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['coral' if x > monthly_avg_season.mean() else 'steelblue' for x in monthly_avg_season]
monthly_avg_season.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_xticklabels(month_names, rotation=45)
ax.set_title('Average Inflation by Month', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Inflation (%)')
ax.axhline(y=monthly_avg_season.mean(), color='black', linestyle='--', label='Average')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("💾 Saved: outputs/figures/seasonal_analysis.png")

---
## Step 10: Interactive Visualisation

In [ ]:
fig = px.line(
    df, x='date', y='close', color='country',
    title='Food Price Index by Country Over Time',
    labels={'close': 'Price Index', 'date': 'Date', 'country': 'Country'}
)
fig.update_layout(
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5),
    hovermode='x unified'
)
fig.show()
print("💡 Tip: Click country names in legend to show/hide them!")

---
## Summary

### Key Findings

| Finding | Description |
|---------|-------------|
| **Price Trend** | Food prices have increased over the study period |
| **Regional Variation** | Inflation varies significantly between countries |
| **Volatility** | Price volatility appears related to inflation |
| **Seasonality** | Some seasonal patterns may exist |

### Charts Created
- `distribution_analysis.png` - Price and inflation distributions
- `time_series_analysis.png` - Trends over time
- `country_comparison.png` - Country rankings
- `correlation_matrix.png` - Variable relationships
- `seasonal_analysis.png` - Monthly patterns

**Next step:** Open `Hypothesis_Testing.ipynb` to statistically validate these patterns!